In [ ]:
import base64
import json

print("Awaiting PAWN payload...")
try:
    # NOTE: Keep the literal "__PAWN_PAYLOAD_B64__" placeholder intact for the backend
    payload_raw = "__PAWN_PAYLOAD_B64__"
    
    # Adding '==' ensures proper byte alignment before decoding.
    padded_payload = payload_raw + "=="
    
    payload = json.loads(base64.b64decode(padded_payload).decode("utf-8"))
    prompt = payload.get("prompt", "a cinematic shot of a highly detailed futuristic city")
    print("Successfully decoded PAWN payload.")
except Exception as e:
    print(f"Warning: Payload decoding failed. Using default prompt. Error: {e}")
    prompt = "a cinematic shot of a highly detailed futuristic city"

print(f"Target Prompt: {prompt}")


In [ ]:
import subprocess
import sys

# Warmup only verifies the connection + primes the slug; it writes a placeholder
# PNG with PIL and needs nothing from this cell. Skipping the (heavy) install keeps
# deploy fast so the warmup run doesn't hold the slug busy and block re-deploys.
if prompt == "warmup":
    print("Warmup prompt detected. Skipping dependency install.")
else:
    print("Installing FLUX dependencies...")
    # No blanket -U -- see image_flux_session/notebook.ipynb for why. FluxPipeline
    # needs diffusers>=0.30.0 (first release with FLUX support); pinned as a floor.
    packages = ["diffusers>=0.30.0", "transformers", "accelerate", "sentencepiece", "protobuf"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
    print("Dependencies installed successfully.")


In [ ]:
import torch
import os

if prompt == "warmup":
    print("Warmup prompt detected. Skipping model loading and GPU inference.")
    from PIL import Image
    img = Image.new("RGB", (1024, 1024), color=(70, 130, 180))
    output_dir = "/kaggle/working"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "out.png")
    img.save(output_path)
    print(f"Warmup output successfully written to {output_path}")
else:
    from diffusers import FluxPipeline

    print("Locating mounted dataset weights...")
    input_base = "/kaggle/input"
    target_dir = None

    # Dynamically find model_index.json to bypass Kaggle's nested dataset folder naming
    for root, dirs, files in os.walk(input_base):
        if "model_index.json" in files:
            target_dir = root
            break

    if not target_dir:
        raise FileNotFoundError("Dataset not found! Ensure dataset_sources is correctly set in kernel-metadata.json.")

    print(f"Loading FLUX pipeline from {target_dir}...")
    # FLUX (~24GB bf16) does not fit one 16GB T4 -> shard across both cards.
    # bf16 is mandatory on Turing T4s (fp16 yields black/NaN images on FLUX).
    try:
        pipe = FluxPipeline.from_pretrained(
            target_dir,
            torch_dtype=torch.bfloat16,
            device_map="balanced",
        )
    except Exception as e:
        # Crash guard against known diffusers multi-GPU device-mismatch / OOM bugs.
        print(f"balanced device_map failed ({e}); retrying with CPU offload.")
        pipe = FluxPipeline.from_pretrained(target_dir, torch_dtype=torch.bfloat16)
        pipe.enable_model_cpu_offload()

    pipe.vae.enable_tiling()
    pipe.set_progress_bar_config(disable=True)

    print("Executing inference...")
    image = pipe(
        prompt=prompt,
        num_inference_steps=4,
        guidance_scale=0.0,
        max_sequence_length=256,
        height=1024,
        width=1024,
    ).images[0]

    output_dir = "/kaggle/working"
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "out.png")
    image.save(output_path)
    print(f"Output successfully written to {output_path} for PAWN backend retrieval.")
